# Cumulative Lie-group splines

This notebook explains how `KernelBase`, `PiecewisePolynomial`, the Irwin–Hall kernels, `CardinalSplineBasis`, and `CumulativeSplineTrajectory` work together to represent differentiable uniform splines in GTSAM.

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/basis/doc/CumulativeSplineTrajectory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass  # Not in Colab

In [ ]:
from math import comb, factorial

import numpy as np
import plotly.graph_objects as go

## Components and data flow

| Component | Role |
|---|---|
| `KernelBase` | Abstract contract for a compactly supported scalar kernel and its derivatives. |
| `PiecewisePolynomial<Order, Pieces>` | Stores each polynomial segment and evaluates exact analytic derivatives. |
| `kernels::IrwinHall*` | Ready-made cardinal B-spline PDFs and cumulative blending kernels. |
| `CardinalSplineBasis` | Converts cumulative cubic-kernel values into ordinary basis weights for the existing `Basis` functors and factors. |
| `CumulativeSplineTrajectory<T>` | Builds expression-valued samples and tangent derivatives when the control points or timestamp are optimization variables. |

For a fixed sample coordinate, use `CardinalSplineBasis` with `BasisFactors`. When time itself is an expression, use `CumulativeSplineTrajectory`; it samples the same cubic kernel but preserves derivatives with respect to time.

## Piecewise-polynomial and Irwin–Hall kernels

On interval $[b_j,b_{j+1}]$, `PiecewisePolynomial` stores

$$p_j(t)=\sum_{k=0}^{d}a_{j,k}t^k,$$

and evaluates any available derivative with the exact power rule. An Irwin–Hall random variable is a sum of independent unit-uniform variables. Its density is a compactly supported cardinal B-spline, and its CDF is the cumulative blending function used by the trajectory. For a sum of $n$ uniforms,

$$F_n(t)=\frac{1}{n!}\sum_{k=0}^{n}(-1)^k {n \choose k}(t-k)_+^n.$$

`IrwinHallCDF2` uses $n=3$, producing the cubic default below.

In [ ]:
def irwin_hall_cdf(order, time):
    time = np.asarray(time, dtype=float)
    result = np.zeros_like(time)
    for k in range(order + 1):
        result += (-1) ** k * comb(order, k) * np.maximum(time - k, 0.0) ** order
    return result / factorial(order)

time = np.linspace(-0.5, 3.5, 401)
cdf = irwin_hall_cdf(3, time)
pdf = np.gradient(cdf, time)

figure = go.Figure()
figure.add_scatter(x=time, y=cdf, name="IrwinHallCDF2")
figure.add_scatter(x=time, y=pdf, name="cubic spline kernel")
figure.update_layout(
    title="Cubic cumulative blending function and its compact kernel",
    xaxis_title="kernel coordinate",
    yaxis_title="value",
    template="plotly_white",
)
figure.show()

## Cumulative interpolation on a Lie group

For control points $T_0,\ldots,T_{N-1}$, define consecutive tangent increments

$$\xi_i=\operatorname{Log}(T_{i-1}^{-1}T_i).$$

The trajectory implemented here uses the first active control point as a local datum:

$$T(t)=T_0\operatorname{Exp}\left(\sum_{i=1}^{N-1}c_i(t)\xi_i\right).$$

Each shifted CDF $c_i(t)$ smoothly activates one increment. Differentiating the piecewise polynomials gives velocity and higher tangent derivatives without finite differences. The density rescales the derivatives from kernel coordinates into physical time.

## Expression-valued trajectory usage

The C++ API accepts key, constant, or computed expressions as control points. A timestamp expression can therefore estimate clock offset jointly with trajectory states.

```cpp
CumulativeSplineTrajectory<Pose3> trajectory(20.0);
for (size_t i = 0; i < poseCount; ++i) {
  trajectory.addControlPoint(Pose3_(Symbol('p', i)));
}

Double_ time(Symbol('t', 0));
Pose3_ pose = trajectory.sampleTrajectory(time, 4.5, 5.5);
Vector6_ velocity =
    trajectory.sampleTrajectoryDerivative(time, 4.5, 5.5, 1);
```

The optional window bounds the plausible time and excludes unrelated control points from the expression graph. The kernel is held by reference, so it must outlive the trajectory object; the exported Irwin–Hall constants have static lifetime.

## Fixed-coordinate basis usage

For a known coordinate, `CardinalSplineBasis` supplies the same cubic interpolation as ordinary dense weights. This connects spline parameters to the reusable factor types in `BasisFactors.h`.

```cpp
Vector controlPoints = (Vector(4) << 0.0, 1.0, 0.0, 1.0).finished();
CardinalSplineBasis::EvaluationFunctor evaluate(4, 2.5);
double value = evaluate(controlPoints);

auto noise = noiseModel::Isotropic::Sigma(1, 0.1);
EvaluationFactor<CardinalSplineBasis> factor(
    Symbol('c', 0), measurement, noise, 4, 2.5);
```

The basis form is linear in scalar or vector control values. The trajectory form is nonlinear but works directly with Lie-group control points and expression-valued time.

## Related geometry adapters

`CartesianProduct<A, B>` reuses `ProductLieGroup` to combine two Lie groups. `AsVectorSpace<Class>` is an explicit local affine adapter for manifold-only types, such as camera calibration classes, when an application needs to include them in a cumulative spline. See the [AsVectorSpace notebook](../../geometry/doc/AsVectorSpace.ipynb) before using that approximation.

## Source

- [CumulativeSplineTrajectory.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/CumulativeSplineTrajectory.h)
- [CardinalSplineBasis.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/CardinalSplineBasis.h)
- [PiecewisePolynomial.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/PiecewisePolynomial.h)
- [IrwinHall.h](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/IrwinHall.h)